# TSViT Colab Reproduction Notebook

This is the cleaned submission notebook for **crop-type semantic segmentation from satellite image time series**.

Main goals:

- Use Colab/A100 for training and evaluation.
- Reuse the original DeepSatModels/TSViT source code in this repository.
- Keep the completed CQ1/CQ2 pipeline for `TSViT`, `UNet3D`, `TViT`, and `STViT`.
- Export qualitative figures in the format `Ground Truth | Prediction | Error/false pixels` for slides and GitHub Pages.

Submission note: execution outputs and sensitive information have been removed. Heavy train/eval cells are guarded by flags so the notebook does not rerun everything accidentally.


## 1. Mount Drive and declare paths

Colab is used for compute only. Dataset files, checkpoints, and exported figures are stored on Google Drive so they survive runtime resets.

Edit `REPO_URL` and the Drive paths if your group uses a different folder structure.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

# ===== User paths =====
REPO_URL = "https://github.com/loanhviet/tsvit-crop-segmentation.git"  # replace loanhviet after creating the group repository
WORK_DIR = "/content/drive/MyDrive/TSViT_BTL"
PROJECT_ROOT = f"{WORK_DIR}/DeepSatModels"
DATA_ROOT = "/content/drive/MyDrive/PASTIS24_extracted/PASTIS24"
CHECKPOINT_ROOT = f"{WORK_DIR}/checkpoints"
RESULTS_ROOT = f"{WORK_DIR}/results"

# Safety flags. Set True only when you intentionally rerun the Colab pipeline.
RUN_SETUP = True
RUN_TRAINING = False
RUN_EVAL = False
EXPORT_FIGURES = False

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)


## 2. Clone source code and install minimal dependencies

The original repository provides most implementations. This project adds cleaned configs, ablation access, and report/demo files. If the repository folder already exists on Drive, the clone step is skipped.


In [ ]:

import os
import subprocess
from pathlib import Path

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

if RUN_SETUP:
    if not Path(PROJECT_ROOT).exists():
        subprocess.run(["git", "clone", REPO_URL, PROJECT_ROOT], check=True)
    else:
        print("Repo already exists:", PROJECT_ROOT)

    subprocess.run([
        "pip", "install", "-q",
        "einops==0.7.0", "timm==0.9.16", "pyyaml", "scikit-learn",
        "pandas", "tensorboard", "torchfcn==1.9.7"
    ], check=True)

os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())


## 3. Check GPU and inspect PASTIS24

PASTIS24 is expected to be stored as 24x24 `.pickle` samples. Each sample contains `img`, `labels`, and `doy`.


In [ ]:

import os
import glob
import pickle
import shutil
import numpy as np
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("DATA_ROOT exists:", os.path.isdir(DATA_ROOT))
print("pickle24x24 exists:", os.path.isdir(f"{DATA_ROOT}/pickle24x24"))
print("fold-paths exists:", os.path.isdir(f"{DATA_ROOT}/fold-paths"))

pickle_files = sorted(glob.glob(f"{DATA_ROOT}/pickle24x24/*.pickle"))
print("num pickle files:", len(pickle_files))
assert pickle_files, "PASTIS24 pickle files were not found. Check DATA_ROOT."

# Copy one sample to /content to avoid Drive I/O errors during inspection.
local_sample = "/content/one_pastis24_sample.pickle"
shutil.copy2(pickle_files[0], local_sample)
with open(local_sample, "rb") as f:
    sample = pickle.load(f, encoding="latin1")

print("sample keys:", sample.keys())
print("img:", sample["img"].shape, sample["img"].dtype)
print("labels:", sample["labels"].shape, sample["labels"].dtype)
print("doy:", sample["doy"].shape, sample["doy"].dtype)

for i in range(sample["labels"].shape[0]):
    u = np.unique(sample["labels"][i])
    print(f"label layer {i}: min={u.min()}, max={u.max()}, n_unique={len(u)}, first={u[:12]}")


## 4. Write `data/datasets.yaml` for Colab

The training code reads dataset locations from `data/datasets.yaml`. This cell writes the PASTIS24 fold protocol using the Drive paths declared above.


In [ ]:

from pathlib import Path
import yaml

datasets_yaml = {
    "MTLCC": {
        "basedir": "...",
        "paths_train": ".../240pkl/train_paths.csv",
        "paths_eval": ".../240pkl/eval_paths.csv",
    },
    "T31TFM_1618": {
        "basedir": "...",
        "paths_train": ".../train_paths.csv",
        "paths_eval": ".../eval_paths.csv",
    },
    "PASTIS24_fold1": {
        "basedir": DATA_ROOT,
        "paths_train": f"{DATA_ROOT}/fold-paths/folds_1_123_paths.csv",
        "paths_eval": f"{DATA_ROOT}/fold-paths/fold_4_paths.csv",
        "paths_test": f"{DATA_ROOT}/fold-paths/fold_5_paths.csv",
    },
    "PASTIS24_fold2": {
        "basedir": DATA_ROOT,
        "paths_train": f"{DATA_ROOT}/fold-paths/folds_2_234_paths.csv",
        "paths_eval": f"{DATA_ROOT}/fold-paths/fold_5_paths.csv",
        "paths_test": f"{DATA_ROOT}/fold-paths/fold_1_paths.csv",
    },
    "PASTIS24_fold3": {
        "basedir": DATA_ROOT,
        "paths_train": f"{DATA_ROOT}/fold-paths/folds_3_345_paths.csv",
        "paths_eval": f"{DATA_ROOT}/fold-paths/fold_1_paths.csv",
        "paths_test": f"{DATA_ROOT}/fold-paths/fold_2_paths.csv",
    },
    "PASTIS24_fold4": {
        "basedir": DATA_ROOT,
        "paths_train": f"{DATA_ROOT}/fold-paths/folds_4_451_paths.csv",
        "paths_eval": f"{DATA_ROOT}/fold-paths/fold_2_paths.csv",
        "paths_test": f"{DATA_ROOT}/fold-paths/fold_3_paths.csv",
    },
    "PASTIS24_fold5": {
        "basedir": DATA_ROOT,
        "paths_train": f"{DATA_ROOT}/fold-paths/folds_5_512_paths.csv",
        "paths_eval": f"{DATA_ROOT}/fold-paths/fold_3_paths.csv",
        "paths_test": f"{DATA_ROOT}/fold-paths/fold_4_paths.csv",
    },
}

out = Path(PROJECT_ROOT) / "data" / "datasets.yaml"
with open(out, "w") as f:
    yaml.safe_dump(datasets_yaml, f, sort_keys=False)

print("updated:", out)


## 5. Patch Colab/PyYAML compatibility

Some Colab environments use a newer PyYAML version, while the original repository calls `yaml.load()` without a Loader. This patch switches config loading to `yaml.safe_load()`.


In [ ]:

from pathlib import Path

utils_path = Path(PROJECT_ROOT) / "utils" / "config_files_utils.py"
text = utils_path.read_text()

if "yaml.safe_load" not in text:
    text = text.replace("from yaml import load, dump", "import yaml")
    text = text.replace("yaml_dict = load(config_file)", "yaml_dict = yaml.safe_load(config_file)")
    text = text.replace(
        "dump(yfile, outfile, default_flow_style=False)",
        "yaml.safe_dump(yfile, outfile, default_flow_style=False, sort_keys=False)",
    )
    utils_path.write_text(text)
    print("patched:", utils_path)
else:
    print("already patched")


## 6. Configs used in the experiments

The group ran four models on the same PASTIS24 fold1 protocol:

- `TSViT`: full model.
- `UNet3D`: CNN baseline.
- `TViT`: ablation without the spatial transformer.
- `STViT`: ablation with spatial-first ordering.

The Colab/A100 configs are included in the repository under `configs/PASTIS24/*_colab_full_a100_opt.yaml`.


In [ ]:

CONFIGS = {
    "TSViT": "configs/PASTIS24/TSViT_fold1_colab_full_a100_opt.yaml",
    "UNet3D": "configs/PASTIS24/UNet3D_fold1_colab_full_a100_opt.yaml",
    "TViT": "configs/PASTIS24/TViT_fold1_colab_full_a100_opt.yaml",
    "STViT": "configs/PASTIS24/STViT_fold1_colab_full_a100_opt.yaml",
}

for name, cfg_path in CONFIGS.items():
    print(name, "->", cfg_path, "exists=", Path(cfg_path).exists())


## 7. Train TSViT and UNet3D

Training is time-consuming, so these cells are disabled by default. Set `RUN_TRAINING = True` in the first code cell only when you intentionally want to reproduce the runs.


In [ ]:

import subprocess

if RUN_TRAINING:
    subprocess.run([
        "python", "train_and_eval/segmentation_training_transf.py",
        "--config", CONFIGS["TSViT"],
        "--device", "0",
    ], check=True)
else:
    print("Skipped TSViT training. Set RUN_TRAINING=True to run.")


In [ ]:

if RUN_TRAINING:
    subprocess.run([
        "python", "train_and_eval/segmentation_training.py",
        "--config_file", CONFIGS["UNet3D"],
        "--gpu_ids", "0",
    ], check=True)
else:
    print("Skipped UNet3D training. Set RUN_TRAINING=True to run.")


## 8. Train ablations: TViT and STViT

These variants use the same dataset, split, and training protocol as TSViT full. Only the architecture is changed to answer CQ2.


In [ ]:

if RUN_TRAINING:
    for model_name in ["TViT", "STViT"]:
        subprocess.run([
            "python", "train_and_eval/segmentation_training_transf.py",
            "--config", CONFIGS[model_name],
            "--device", "0",
        ], check=True)
else:
    print("Skipped TViT/STViT training. Set RUN_TRAINING=True to run.")


## 9. Evaluate checkpoints on the test split

Fill in real checkpoint paths in `CHECKPOINTS` before evaluation. For the completed runs:

- TSViT uses the best checkpoint from the TSViT checkpoint folder.
- UNet3D uses the selected checkpoint from the UNet3D training run.
- TViT/STViT use the corresponding best checkpoints.


In [ ]:

CHECKPOINTS = {
    "TSViT": f"{CHECKPOINT_ROOT}/TSViT_fold1_full_a100_opt/best.pth",
    "UNet3D": f"{CHECKPOINT_ROOT}/UNet3D_fold1_full_a100_opt/UNet3D_fold1_full_a100_opt_11_12000.pth",
    "TViT": f"{CHECKPOINT_ROOT}/TViT_fold1_full_a100_opt/best.pth",
    "STViT": f"{CHECKPOINT_ROOT}/STViT_fold1_full_a100_opt/best.pth",
}

for name, ckpt in CHECKPOINTS.items():
    print(name, "->", ckpt, "exists=", Path(ckpt).exists())


In [ ]:

import numpy as np
import torch

from models import get_model
from utils.torch_utils import get_device, load_from_checkpoint
from utils.config_files_utils import read_yaml
from data.PASTIS24.dataloader import get_dataloader
from data.PASTIS24.data_transforms import PASTIS_segmentation_transform
from data import get_loss_data_input
from metrics.loss_functions import get_loss
from metrics.numpy_metrics import get_classification_metrics


def evaluate_checkpoint(model_name, cfg_path, ckpt_path):
    config = read_yaml(cfg_path)
    config["local_device_ids"] = [0]

    device = get_device([0], allow_cpu=False)
    test_paths = f"{DATA_ROOT}/fold-paths/fold_5_paths.csv"

    test_loader = get_dataloader(
        paths_file=test_paths,
        root_dir=DATA_ROOT,
        transform=PASTIS_segmentation_transform(config["MODEL"], is_training=False),
        batch_size=config["DATASETS"]["test"]["batch_size"],
        shuffle=False,
        num_workers=config["DATASETS"]["test"]["num_workers"],
    )

    net = get_model(config, device)
    load_from_checkpoint(net, ckpt_path, partial_restore=False, device=device)
    net.to(device)
    net.eval()

    loss_input_fn = get_loss_data_input(config)
    loss_fn = get_loss(config, device, reduction=None)

    predicted_all, labels_all, losses_all = [], [], []

    with torch.no_grad():
        for step, sample in enumerate(test_loader):
            if step % 50 == 0:
                print(f"{model_name}: test step {step}/{len(test_loader)}")

            logits = net(sample["inputs"].to(device))
            logits = logits.permute(0, 2, 3, 1)
            _, predicted = torch.max(logits.data, -1)

            ground_truth = loss_input_fn(sample, device)
            loss = loss_fn(logits, ground_truth)
            target, mask = ground_truth

            if mask is not None:
                predicted_all.append(predicted.view(-1)[mask.view(-1)].cpu().numpy())
                labels_all.append(target.view(-1)[mask.view(-1)].cpu().numpy())
            else:
                predicted_all.append(predicted.view(-1).cpu().numpy())
                labels_all.append(target.view(-1).cpu().numpy())

            losses_all.append(loss.view(-1).cpu().detach().numpy())

    predicted_classes = np.concatenate(predicted_all)
    target_classes = np.concatenate(labels_all)
    losses = np.concatenate(losses_all)

    metrics = get_classification_metrics(
        predicted=predicted_classes,
        labels=target_classes,
        n_classes=config["MODEL"]["num_classes"],
        unk_masks=None,
    )

    micro_acc, micro_precision, micro_recall, micro_f1, micro_iou = metrics["micro"]
    macro_acc, macro_precision, macro_recall, macro_f1, macro_iou = metrics["macro"]

    return {
        "model": model_name,
        "loss": float(losses.mean()),
        "oa": float(micro_acc),
        "miou": float(macro_iou),
        "f1_macro": float(macro_f1),
        "precision_macro": float(macro_precision),
        "recall_macro": float(macro_recall),
        "unique_pred_labels": np.unique(predicted_classes).tolist(),
    }


if RUN_EVAL:
    eval_rows = []
    for model_name in ["TSViT", "UNet3D", "TViT", "STViT"]:
        eval_rows.append(evaluate_checkpoint(model_name, CONFIGS[model_name], CHECKPOINTS[model_name]))
    eval_rows
else:
    print("Skipped evaluation. Set RUN_EVAL=True to run.")


## 10. Completed results

### CQ1: TSViT vs UNet3D

| Model | Loss | OA | mIoU | F1-macro | Precision | Recall | Params |
|---|---:|---:|---:|---:|---:|---:|---:|
| TSViT | 0.6022 | 0.8271 | 0.6361 | 0.7623 | 0.7841 | 0.7465 | 1.657M |
| UNet3D | 0.6281 | 0.8011 | 0.5720 | 0.7045 | 0.7573 | 0.6711 | 6.177M |

CQ1 conclusion: TSViT is better than UNet3D on mIoU and macro F1 while using fewer parameters.

### CQ2: TSViT ablation

| Model | Loss | OA | mIoU | F1-macro | Precision | Recall |
|---|---:|---:|---:|---:|---:|---:|
| TSViT full | 0.6022 | 0.8271 | 0.6361 | 0.7623 | 0.7841 | 0.7465 |
| TViT, no spatial | 1.0591 | 0.6685 | 0.3713 | 0.5267 | 0.5654 | 0.5049 |
| STViT, spatial-first | 0.7707 | 0.7939 | 0.5572 | 0.6921 | 0.7424 | 0.6660 |

Delta mIoU compared with TSViT full:

- TViT: `-0.2648`, the largest drop, showing that spatial modeling is critical.
- STViT: `-0.0789`, better than TViT but still below TSViT full, showing that temporal-first ordering works better in this setup.


## 11. Export qualitative figures

This cell exports qualitative figures for slides and the GitHub Pages demo. Main formats:

- CQ1: `Ground Truth | UNet3D | TSViT`.
- CQ2: the same logic can be used for `Ground Truth | TViT | STViT | TSViT`.

Wrong pixels are marked with `x`, following the style used in TSViT qualitative visualizations.


In [ ]:

import csv
import os
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

CLASS_COLORS = np.array([
    [0.10, 0.10, 0.10], [0.85, 0.20, 0.20], [0.20, 0.55, 0.90],
    [0.25, 0.70, 0.30], [0.95, 0.65, 0.15], [0.55, 0.35, 0.80],
    [0.10, 0.75, 0.75], [0.90, 0.45, 0.75], [0.55, 0.45, 0.25],
    [0.60, 0.80, 0.20], [0.35, 0.35, 0.85], [0.95, 0.35, 0.10],
    [0.20, 0.80, 0.55], [0.75, 0.25, 0.55], [0.40, 0.65, 0.95],
    [0.90, 0.85, 0.20], [0.45, 0.25, 0.15], [0.55, 0.55, 0.55],
    [0.15, 0.45, 0.20], [1.00, 1.00, 1.00],
])
CLASS_CMAP = ListedColormap(CLASS_COLORS)


def sample_miou(pred, gt, mask, n_classes=19):
    ious = []
    for cls in range(n_classes):
        p = (pred == cls) & mask
        g = (gt == cls) & mask
        union = p | g
        if union.sum() == 0:
            continue
        ious.append((p & g).sum() / union.sum())
    return float(np.mean(ious)) if ious else 0.0


def add_wrong_marks(ax, pred, gt, mask, max_marks=180):
    wrong = (pred != gt) & mask
    ys, xs = np.where(wrong)
    if len(xs) == 0:
        return int(wrong.sum())
    if len(xs) > max_marks:
        pick = np.linspace(0, len(xs) - 1, max_marks).astype(int)
        xs, ys = xs[pick], ys[pick]
    ax.scatter(xs, ys, marker="x", c="black", s=8, linewidths=0.45)
    return int(wrong.sum())


In [ ]:

def load_model_for_prediction(model_name):
    config = read_yaml(CONFIGS[model_name])
    config["local_device_ids"] = [0]
    device = get_device([0], allow_cpu=False)
    net = get_model(config, device)
    load_from_checkpoint(net, CHECKPOINTS[model_name], partial_restore=False, device=device)
    net.to(device)
    net.eval()
    return net, config, device


def export_cq1_figures(target_indices=(0, 3099, 6199, 9299, 12399)):
    out_dir = f"{RESULTS_ROOT}/figures/cq1_tsvit_vs_unet3d_paper_style"
    os.makedirs(out_dir, exist_ok=True)

    tsvit, tsvit_cfg, device = load_model_for_prediction("TSViT")
    unet, _, _ = load_model_for_prediction("UNet3D")

    test_loader = get_dataloader(
        paths_file=f"{DATA_ROOT}/fold-paths/fold_5_paths.csv",
        root_dir=DATA_ROOT,
        transform=PASTIS_segmentation_transform(tsvit_cfg["MODEL"], is_training=False),
        batch_size=32,
        shuffle=False,
        num_workers=8,
    )

    target_indices = set(target_indices)
    saved_rows = []

    with torch.no_grad():
        for step, sample in enumerate(test_loader):
            inputs = sample["inputs"].to(device)
            pred_tsvit = torch.argmax(tsvit(inputs), dim=1).cpu().numpy()
            pred_unet = torch.argmax(unet(inputs), dim=1).cpu().numpy()
            labels = sample["labels"].squeeze(-1).cpu().numpy().astype(np.int64)
            masks = sample["unk_masks"].squeeze(-1).cpu().numpy().astype(bool)

            for i in range(inputs.shape[0]):
                global_idx = step * test_loader.batch_size + i
                if global_idx not in target_indices:
                    continue

                gt = labels[i]
                mask = masks[i]
                gt_vis = gt.copy()
                unet_vis = pred_unet[i].copy()
                tsvit_vis = pred_tsvit[i].copy()
                gt_vis[~mask] = 19
                unet_vis[~mask] = 19
                tsvit_vis[~mask] = 19

                fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.2), dpi=220)
                axes[0].imshow(gt_vis, cmap=CLASS_CMAP, vmin=0, vmax=19, interpolation="nearest")
                axes[0].set_title("Ground truth", fontsize=10)

                axes[1].imshow(unet_vis, cmap=CLASS_CMAP, vmin=0, vmax=19, interpolation="nearest")
                axes[1].set_title("UNet3D", fontsize=10)
                unet_wrong = add_wrong_marks(axes[1], pred_unet[i], gt, mask)

                axes[2].imshow(tsvit_vis, cmap=CLASS_CMAP, vmin=0, vmax=19, interpolation="nearest")
                axes[2].set_title("TSViT", fontsize=10)
                tsvit_wrong = add_wrong_marks(axes[2], pred_tsvit[i], gt, mask)

                for ax in axes:
                    ax.set_xticks([])
                    ax.set_yticks([])
                    ax.set_frame_on(False)

                fig.suptitle(f"PASTIS24 fold1 test sample #{global_idx}", fontsize=10)
                plt.tight_layout()
                out_path = os.path.join(out_dir, f"cq1_compare_sample_{global_idx:05d}.png")
                plt.savefig(out_path, bbox_inches="tight")
                plt.close(fig)

                saved_rows.append({
                    "global_idx": global_idx,
                    "unet_miou": sample_miou(pred_unet[i], gt, mask),
                    "unet_wrong_pixels": unet_wrong,
                    "tsvit_miou": sample_miou(pred_tsvit[i], gt, mask),
                    "tsvit_wrong_pixels": tsvit_wrong,
                    "figure_path": out_path,
                })

            if len(saved_rows) >= len(target_indices):
                break

    csv_path = os.path.join(out_dir, "cq1_compare_samples_summary.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=saved_rows[0].keys())
        writer.writeheader()
        writer.writerows(saved_rows)

    return saved_rows


if EXPORT_FIGURES:
    export_cq1_figures()
else:
    print("Skipped qualitative export. Set EXPORT_FIGURES=True to run.")


## 12. Presentation notes

- This notebook documents the Colab pipeline: setup source, declare dataset paths, train/evaluate, and export figures.
- The submitted results focus on the two completed experiment groups: baseline comparison and ablation study.
- Qualitative figures from the PASTIS24 test split are used to show how model predictions become segmentation maps.
- Dataset and checkpoint artifacts are not committed to GitHub because they are large; only code, configs, the cleaned notebook, and the GitHub Pages demo are submitted.
